# Silver to Gold
Builds the Gold layer: a **star schema** that Power BI can read directly.

| Kind | Table | One row is... | Primary key |
|---|---|---|---|
| Dimension | `dim_repo` | one repository | `repo_key` |
| Dimension | `dim_date` | one calendar day | `date_key` (yyyymmdd) |
| Dimension | `dim_contributor` | one (hashed) person | `author_key` |
| Fact | `fact_commit` | one commit | `sha` |
| Fact | `fact_issue` | one issue | `issue_id` |
| Fact | `fact_pull_request` | one pull request | `pr_id` |
| Aggregate fact | `fact_repo_daily_activity` | one repo on one day | `repo_key` + `date_key` |

Every write is a `MERGE INTO`, so the notebook is safe to run again.
Every table gets one row in `pipeline_execution_logs` with
`layer = Silver-to-Gold`, the rows aggregated, and the seconds it took.

In [0]:
%run ./00_setup_and_logging

# Setup and logging
This notebook does two jobs:
1. Makes the schema (database) and all tables, if they don't exist yet.
2. Gives small helper functions that write one row to the log table every time a step runs.

Run this notebook by itself once. Other notebooks call it with `%run`.

Using workspace.dav_project


## Bronze tables
One table per entity (commits, issues, pulls), plus one quarantine table for
records that could not even be given a key. Every table has `load_timestamp`.

ok: bronze_commits
ok: bronze_issues
ok: bronze_pulls


ok: bronze_quarantine


...

ok: pipeline_execution_logs


## Phase 3 upgrade of the log table
The log table already exists from Phase 2, so `CREATE TABLE IF NOT EXISTS`
above does not change it. This cell adds the two new columns to an old
table, only if they are missing. Safe to run any number of times.
- `rows_aggregated`: how many source rows were read and rolled up (Silver to Gold).
- `duration_seconds`: how long the step took.

column already there: rows_aggregated
column already there: duration_seconds


## Small helper functions
`new_run_id()` makes one ID for one notebook run.
`log_run(...)` writes one row to the log table. Call it even when a step fails.
`get_merge_metrics(...)` reads how many rows a MERGE just inserted and updated.

helpers ready: new_run_id, now_utc, log_run, get_merge_metrics


In [0]:
dbutils.widgets.text("silver_since", "", "Only Silver rows loaded since (YYYY-MM-DDTHH:MM:SSZ, blank = all)")
dbutils.widgets.text("batch_id", "", "Batch name for the log (blank = auto)")
dbutils.widgets.dropdown("run_optimize", "yes", ["yes", "no"], "Run OPTIMIZE at the end?")

SILVER_SINCE = dbutils.widgets.get("silver_since").strip() or None
RUN_OPTIMIZE = dbutils.widgets.get("run_optimize") == "yes"
RUN_ID = new_run_id()
BATCH_NAME = dbutils.widgets.get("batch_id").strip() or f"gold_{now_utc().strftime('%Y%m%dT%H%M%S')}"
FAILED_STEPS = []

T = lambda name: f"{CATALOG}.{SCHEMA}.{name}"   # full table name
print("run_id:", RUN_ID, "| silver_since:", SILVER_SINCE or "(all of Silver)", "| optimize:", RUN_OPTIMIZE)

run_id: afd6ddda-da8c-4082-9802-bd463d208039 | silver_since: (all of Silver) | optimize: True


## 1. Create the Gold tables (explicit columns, no guessing)
`CLUSTER BY` turns on **Liquid Clustering**: Delta keeps rows with the same
`repo_key` / `date_key` stored close together, so dashboard filters on a repo or
a date range read fewer files. We did not use `PARTITIONED BY`: the data is
small, and partitioning small data makes lots of tiny files.

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {T('dim_repo')} (
  repo_key        BIGINT  NOT NULL,
  repo_full_name  STRING  NOT NULL,
  owner           STRING,
  repo_name       STRING
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {T('dim_date')} (
  date_key       INT     NOT NULL,
  full_date      DATE    NOT NULL,
  year           INT,
  quarter        INT,
  month          INT,
  month_name     STRING,
  year_month     STRING,
  day_of_month   INT,
  day_of_week    INT,
  day_name       STRING,
  week_of_year   INT,
  is_weekend     BOOLEAN
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {T('dim_contributor')} (
  author_key      STRING  NOT NULL,
  first_seen_at   TIMESTAMP
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {T('fact_commit')} (
  sha               STRING    NOT NULL,
  repo_key          BIGINT    NOT NULL,
  author_key        STRING    NOT NULL,
  committer_key     STRING    NOT NULL,
  commit_date_key   INT,
  committed_at      TIMESTAMP,
  message_length    INT,
  is_merge_commit   BOOLEAN,
  load_timestamp    TIMESTAMP
) USING DELTA
CLUSTER BY (repo_key, commit_date_key)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {T('fact_issue')} (
  issue_id            STRING    NOT NULL,
  repo_key            BIGINT    NOT NULL,
  author_key          STRING    NOT NULL,
  created_date_key    INT,
  closed_date_key     INT,
  issue_number        INT,
  state               STRING,
  is_closed           BOOLEAN,
  comment_count       INT,
  time_to_close_hours DOUBLE,
  created_at          TIMESTAMP,
  closed_at           TIMESTAMP,
  load_timestamp      TIMESTAMP
) USING DELTA
CLUSTER BY (repo_key, created_date_key)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {T('fact_pull_request')} (
  pr_id               STRING    NOT NULL,
  repo_key            BIGINT    NOT NULL,
  author_key          STRING    NOT NULL,
  created_date_key    INT,
  closed_date_key     INT,
  merged_date_key     INT,
  pr_number           INT,
  state               STRING,
  is_merged           BOOLEAN,
  is_draft            BOOLEAN,
  base_branch         STRING,
  time_to_merge_hours DOUBLE,
  created_at          TIMESTAMP,
  closed_at           TIMESTAMP,
  merged_at           TIMESTAMP,
  load_timestamp      TIMESTAMP
) USING DELTA
CLUSTER BY (repo_key, created_date_key)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {T('fact_repo_daily_activity')} (
  repo_key             BIGINT  NOT NULL,
  date_key             INT     NOT NULL,
  commits              BIGINT,
  merge_commits        BIGINT,
  active_contributors  BIGINT,
  issues_opened        BIGINT,
  issues_closed        BIGINT,
  prs_opened           BIGINT,
  prs_merged           BIGINT,
  updated_at           TIMESTAMP
) USING DELTA
CLUSTER BY (repo_key, date_key)
""")
print("ok: 7 Gold tables exist")

ok: 7 Gold tables exist


## 2. Primary keys and foreign keys (the data dictionary, inside the database)
Databricks stores these as **informational** constraints: they are written down
in Unity Catalog (you can see them in Catalog Explorer) but not enforced.
The MERGE keys are what really keep the keys unique.
If a constraint can't be added, this cell prints a warning and carries on.

In [0]:
CONSTRAINTS = [
    ("dim_repo", "pk_dim_repo", "PRIMARY KEY (repo_key)"),
    ("dim_date", "pk_dim_date", "PRIMARY KEY (date_key)"),
    ("dim_contributor", "pk_dim_contributor", "PRIMARY KEY (author_key)"),
    ("fact_commit", "pk_fact_commit", "PRIMARY KEY (sha)"),
    ("fact_issue", "pk_fact_issue", "PRIMARY KEY (issue_id)"),
    ("fact_pull_request", "pk_fact_pull_request", "PRIMARY KEY (pr_id)"),
    ("fact_repo_daily_activity", "pk_fact_repo_daily", "PRIMARY KEY (repo_key, date_key)"),
    ("fact_commit", "fk_commit_repo", f"FOREIGN KEY (repo_key) REFERENCES {T('dim_repo')}"),
    ("fact_commit", "fk_commit_date", f"FOREIGN KEY (commit_date_key) REFERENCES {T('dim_date')}"),
    ("fact_commit", "fk_commit_author", f"FOREIGN KEY (author_key) REFERENCES {T('dim_contributor')}"),
    ("fact_issue", "fk_issue_repo", f"FOREIGN KEY (repo_key) REFERENCES {T('dim_repo')}"),
    ("fact_issue", "fk_issue_created", f"FOREIGN KEY (created_date_key) REFERENCES {T('dim_date')}"),
    ("fact_issue", "fk_issue_author", f"FOREIGN KEY (author_key) REFERENCES {T('dim_contributor')}"),
    ("fact_pull_request", "fk_pr_repo", f"FOREIGN KEY (repo_key) REFERENCES {T('dim_repo')}"),
    ("fact_pull_request", "fk_pr_created", f"FOREIGN KEY (created_date_key) REFERENCES {T('dim_date')}"),
    ("fact_pull_request", "fk_pr_author", f"FOREIGN KEY (author_key) REFERENCES {T('dim_contributor')}"),
    ("fact_repo_daily_activity", "fk_daily_repo", f"FOREIGN KEY (repo_key) REFERENCES {T('dim_repo')}"),
    ("fact_repo_daily_activity", "fk_daily_date", f"FOREIGN KEY (date_key) REFERENCES {T('dim_date')}"),
]

for table, name, definition in CONSTRAINTS:
    try:
        spark.sql(f"ALTER TABLE {T(table)} ADD CONSTRAINT {name} {definition}")
        print(f"added: {table}.{name}")
    except Exception as e:
        msg = str(e)
        if "already exists" in msg.lower() or "CONSTRAINT_ALREADY_EXISTS" in msg:
            print(f"already there: {table}.{name}")
        else:
            print(f"WARNING: could not add {table}.{name}: {msg[:160]}")

already there: dim_repo.pk_dim_repo
already there: dim_date.pk_dim_date
already there: dim_contributor.pk_dim_contributor
already there: fact_commit.pk_fact_commit
already there: fact_issue.pk_fact_issue
already there: fact_pull_request.pk_fact_pull_request
already there: fact_repo_daily_activity.pk_fact_repo_daily
already there: fact_commit.fk_commit_repo
already there: fact_commit.fk_commit_date
already there: fact_commit.fk_commit_author
already there: fact_issue.fk_issue_repo
already there: fact_issue.fk_issue_created
already there: fact_issue.fk_issue_author
already there: fact_pull_request.fk_pr_repo
already there: fact_pull_request.fk_pr_created
already there: fact_pull_request.fk_pr_author
already there: fact_repo_daily_activity.fk_daily_repo
already there: fact_repo_daily_activity.fk_daily_date


## 3. The SELECT for each Gold table
Each table is a plain SQL `SELECT` over Silver. They are kept as text so the
same SQL is used for the MERGE and for `EXPLAIN` in the report.

- `repo_key` is `xxhash64(repo)`: the same repo always gets the same number.
- `*_date_key` is the day as a number like `20261024`.
- A missing author becomes the member `unknown`, so no fact row has a broken link.
- Facts only read Silver rows loaded since `silver_since` (incremental). The
  small aggregate table is rebuilt from the Gold facts each time.

In [0]:
def since_clause(col="load_timestamp"):
    return f"WHERE {col} >= TIMESTAMP '{SILVER_SINCE.replace('T', ' ').replace('Z', '')}'" if SILVER_SINCE else ""


DATE_KEY = lambda c: f"CAST(date_format({c}, 'yyyyMMdd') AS INT)"


def sql_dim_repo():
    return f"""
        SELECT xxhash64(repo) AS repo_key,
               repo           AS repo_full_name,
               split(repo, '/')[0] AS owner,
               split(repo, '/')[1] AS repo_name
        FROM (SELECT repo FROM {T('silver_commit')}
              UNION SELECT repo FROM {T('silver_issue')}
              UNION SELECT repo FROM {T('silver_pull_request')})
        WHERE repo IS NOT NULL"""


def sql_dim_date():
    return """
        SELECT CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key,
               d AS full_date,
               year(d) AS year, quarter(d) AS quarter, month(d) AS month,
               date_format(d, 'MMMM') AS month_name,
               date_format(d, 'yyyy-MM') AS year_month,
               day(d) AS day_of_month,
               dayofweek(d) AS day_of_week,
               date_format(d, 'EEEE') AS day_name,
               weekofyear(d) AS week_of_year,
               dayofweek(d) IN (1, 7) AS is_weekend
        FROM (SELECT explode(sequence(DATE'2008-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS d)"""


def sql_dim_contributor():
    return f"""
        SELECT author_key, MIN(seen_at) AS first_seen_at
        FROM (
            SELECT author_key, committed_at AS seen_at FROM {T('silver_commit')}
            UNION ALL SELECT committer_key, committed_at FROM {T('silver_commit')}
            UNION ALL SELECT author_key, created_at FROM {T('silver_issue')}
            UNION ALL SELECT author_key, created_at FROM {T('silver_pull_request')}
        )
        WHERE author_key IS NOT NULL
        GROUP BY author_key
        UNION ALL
        SELECT 'unknown', CAST(NULL AS TIMESTAMP)"""


def sql_fact_commit():
    return f"""
        SELECT sha,
               xxhash64(repo) AS repo_key,
               coalesce(author_key, 'unknown') AS author_key,
               coalesce(committer_key, 'unknown') AS committer_key,
               {DATE_KEY('committed_at')} AS commit_date_key,
               committed_at, message_length, is_merge_commit, load_timestamp
        FROM {T('silver_commit')}
        {since_clause()}"""


def sql_fact_issue():
    return f"""
        SELECT issue_id,
               xxhash64(repo) AS repo_key,
               coalesce(author_key, 'unknown') AS author_key,
               {DATE_KEY('created_at')} AS created_date_key,
               {DATE_KEY('closed_at')}  AS closed_date_key,
               number AS issue_number,
               state,
               closed_at IS NOT NULL AS is_closed,
               comment_count, time_to_close_hours,
               created_at, closed_at, load_timestamp
        FROM {T('silver_issue')}
        {since_clause()}"""


def sql_fact_pull_request():
    return f"""
        SELECT pr_id,
               xxhash64(repo) AS repo_key,
               coalesce(author_key, 'unknown') AS author_key,
               {DATE_KEY('created_at')} AS created_date_key,
               {DATE_KEY('closed_at')}  AS closed_date_key,
               {DATE_KEY('merged_at')}  AS merged_date_key,
               number AS pr_number,
               state, is_merged, is_draft, base_branch, time_to_merge_hours,
               created_at, closed_at, merged_at, load_timestamp
        FROM {T('silver_pull_request')}
        {since_clause()}"""


def sql_fact_repo_daily_activity():
    """Rolls the three Gold facts up to one row per repo per day."""
    return f"""
        WITH c AS (
            SELECT repo_key, commit_date_key AS date_key,
                   count(*) AS commits,
                   sum(CASE WHEN is_merge_commit THEN 1 ELSE 0 END) AS merge_commits,
                   count(DISTINCT author_key) AS active_contributors
            FROM {T('fact_commit')} WHERE commit_date_key IS NOT NULL GROUP BY repo_key, commit_date_key),
        io AS (
            SELECT repo_key, created_date_key AS date_key, count(*) AS issues_opened
            FROM {T('fact_issue')} WHERE created_date_key IS NOT NULL GROUP BY repo_key, created_date_key),
        ic AS (
            SELECT repo_key, closed_date_key AS date_key, count(*) AS issues_closed
            FROM {T('fact_issue')} WHERE closed_date_key IS NOT NULL GROUP BY repo_key, closed_date_key),
        po AS (
            SELECT repo_key, created_date_key AS date_key, count(*) AS prs_opened
            FROM {T('fact_pull_request')} WHERE created_date_key IS NOT NULL GROUP BY repo_key, created_date_key),
        pm AS (
            SELECT repo_key, merged_date_key AS date_key, count(*) AS prs_merged
            FROM {T('fact_pull_request')} WHERE merged_date_key IS NOT NULL GROUP BY repo_key, merged_date_key),
        k AS (
            SELECT repo_key, date_key FROM c UNION SELECT repo_key, date_key FROM io
            UNION SELECT repo_key, date_key FROM ic UNION SELECT repo_key, date_key FROM po
            UNION SELECT repo_key, date_key FROM pm)
        SELECT k.repo_key, k.date_key,
               coalesce(c.commits, 0)              AS commits,
               coalesce(c.merge_commits, 0)        AS merge_commits,
               coalesce(c.active_contributors, 0)  AS active_contributors,
               coalesce(io.issues_opened, 0)       AS issues_opened,
               coalesce(ic.issues_closed, 0)       AS issues_closed,
               coalesce(po.prs_opened, 0)          AS prs_opened,
               coalesce(pm.prs_merged, 0)          AS prs_merged,
               current_timestamp()                 AS updated_at
        FROM k
        LEFT JOIN c  ON k.repo_key = c.repo_key  AND k.date_key = c.date_key
        LEFT JOIN io ON k.repo_key = io.repo_key AND k.date_key = io.date_key
        LEFT JOIN ic ON k.repo_key = ic.repo_key AND k.date_key = ic.date_key
        LEFT JOIN po ON k.repo_key = po.repo_key AND k.date_key = po.date_key
        LEFT JOIN pm ON k.repo_key = pm.repo_key AND k.date_key = pm.date_key"""


## 4. Run each table: MERGE, then log it
`rows_aggregated` = how many source rows the MERGE read (Delta reports it as
`numSourceRows`). `duration_seconds` is worked out inside `log_run`.
If one table fails, it is logged as FAILED and the next one still runs.

In [0]:
def merge_stats(table):
    """Counts from the latest MERGE on this table."""
    h = spark.sql(f"DESCRIBE HISTORY {T(table)} LIMIT 1").collect()[0]
    m = h["operationMetrics"] or {}
    return {
        "inserted": int(m.get("numTargetRowsInserted", 0)),
        "updated": int(m.get("numTargetRowsUpdated", 0)),
        "source": int(m.get("numSourceRows", 0)),
    }


def run_merge(table, select_sql, key_cols, when_matched, note=None):
    """MERGE one Gold table. when_matched is the full 'WHEN MATCHED ...' text, or '' for insert-only."""
    start = now_utc()
    params = {"silver_since": SILVER_SINCE, "batch": BATCH_NAME}
    try:
        on = " AND ".join(f"t.{c} = s.{c}" for c in key_cols)
        spark.sql(f"""
            MERGE INTO {T(table)} t
            USING ({select_sql}) s
            ON {on}
            {when_matched}
            WHEN NOT MATCHED THEN INSERT *
        """)
        m = merge_stats(table)
        log_run(RUN_ID, "Silver-to-Gold", table, params, start, now_utc(), "SUCCESS",
                rows_inserted=m["inserted"], rows_updated=m["updated"], rows_aggregated=m["source"])
        return m
    except Exception as e:
        log_run(RUN_ID, "Silver-to-Gold", table, params, start, now_utc(), "FAILED",
                error_message=str(e)[:900])
        print(f"FAILED: {table}: {e}")
        FAILED_STEPS.append(f"{table}: {str(e)[:200]}")
        return None


UPDATE_ALL = "WHEN MATCHED THEN UPDATE SET *"
results = {}

# Dimensions first, because the facts point at them.
results["dim_repo"] = run_merge("dim_repo", sql_dim_repo(), ["repo_key"], "")
results["dim_date"] = run_merge("dim_date", sql_dim_date(), ["date_key"], "")
results["dim_contributor"] = run_merge(
    "dim_contributor", sql_dim_contributor(), ["author_key"],
    # only change a row if we found an EARLIER first-seen time (can happen in a backfill)
    "WHEN MATCHED AND s.first_seen_at < t.first_seen_at THEN UPDATE SET t.first_seen_at = s.first_seen_at")

# Facts
results["fact_commit"] = run_merge("fact_commit", sql_fact_commit(), ["sha"], UPDATE_ALL)
results["fact_issue"] = run_merge("fact_issue", sql_fact_issue(), ["issue_id"], UPDATE_ALL)
results["fact_pull_request"] = run_merge("fact_pull_request", sql_fact_pull_request(), ["pr_id"], UPDATE_ALL)

# Aggregate fact: rebuilt from the Gold facts. Only rows whose numbers changed are updated.
results["fact_repo_daily_activity"] = run_merge(
    "fact_repo_daily_activity", sql_fact_repo_daily_activity(), ["repo_key", "date_key"],
    """WHEN MATCHED AND NOT (t.commits <=> s.commits AND t.merge_commits <=> s.merge_commits
                            AND t.active_contributors <=> s.active_contributors
                            AND t.issues_opened <=> s.issues_opened AND t.issues_closed <=> s.issues_closed
                            AND t.prs_opened <=> s.prs_opened AND t.prs_merged <=> s.prs_merged)
       THEN UPDATE SET *""")

for name, m in results.items():
    print(f"{name:28s}", m)

[log] Silver-to-Gold / dim_repo -> SUCCESS (inserted=1, updated=0, quarantined=0, aggregated=9, seconds=4.5)
[log] Silver-to-Gold / dim_date -> SUCCESS (inserted=0, updated=0, quarantined=0, aggregated=8401, seconds=3.9)
[log] Silver-to-Gold / dim_contributor -> SUCCESS (inserted=3, updated=0, quarantined=0, aggregated=8128, seconds=7.0)
[log] Silver-to-Gold / fact_commit -> SUCCESS (inserted=3, updated=13840, quarantined=0, aggregated=13843, seconds=5.5)
[log] Silver-to-Gold / fact_issue -> SUCCESS (inserted=3, updated=8835, quarantined=0, aggregated=8838, seconds=5.2)
[log] Silver-to-Gold / fact_pull_request -> SUCCESS (inserted=2, updated=18156, quarantined=0, aggregated=18158, seconds=6.8)
[log] Silver-to-Gold / fact_repo_daily_activity -> SUCCESS (inserted=2, updated=3, quarantined=0, aggregated=8599, seconds=8.8)
dim_repo                     {'inserted': 1, 'updated': 0, 'source': 9}
dim_date                     {'inserted': 0, 'updated': 0, 'source': 8401}
dim_contributor       

## 5. Audit views for the dashboard
Power BI reads these like any table. They hide the raw JSON (it can hold names),
and only show counts and reasons.

In [0]:
try:
    spark.sql(f"""
    CREATE OR REPLACE VIEW {T('vw_audit_pipeline_runs')} AS
    SELECT run_id, layer, entity, status,
           rows_inserted, rows_updated, rows_quarantined, rows_aggregated,
           duration_seconds, start_time, end_time,
           CAST(start_time AS DATE) AS run_date,
           error_message
    FROM {T('pipeline_execution_logs')}""")

    spark.sql(f"""
    CREATE OR REPLACE VIEW {T('vw_audit_quarantine')} AS
    SELECT 'Bronze' AS layer, entity_type, source_repo AS repo, reason,
           CAST(load_timestamp AS DATE) AS load_date, count(*) AS quarantined_rows
    FROM {T('bronze_quarantine')}
    GROUP BY entity_type, source_repo, reason, CAST(load_timestamp AS DATE)
    UNION ALL
    SELECT 'Silver', entity_type, repo, reason,
           CAST(load_timestamp AS DATE), count(*)
    FROM {T('silver_quarantine')}
    GROUP BY entity_type, repo, reason, CAST(load_timestamp AS DATE)""")
    print("ok: vw_audit_pipeline_runs, vw_audit_quarantine")
except Exception as e:
    print("FAILED: audit views:", e)
    FAILED_STEPS.append(f"audit views: {str(e)[:200]}")

ok: vw_audit_pipeline_runs, vw_audit_quarantine


## 6. OPTIMIZE (optional)
On a table with `CLUSTER BY`, `OPTIMIZE` joins small files together **and** puts
rows in clustered order. Run it after loads, not before every query.

In [0]:
if RUN_OPTIMIZE:
    for table in ["fact_commit", "fact_issue", "fact_pull_request", "fact_repo_daily_activity"]:
        start = now_utc()
        try:
            spark.sql(f"OPTIMIZE {T(table)}")
            log_run(RUN_ID, "Silver-to-Gold", f"optimize:{table}", {"batch": BATCH_NAME},
                    start, now_utc(), "SUCCESS")
            print("optimized:", table)
        except Exception as e:
            log_run(RUN_ID, "Silver-to-Gold", f"optimize:{table}", {"batch": BATCH_NAME},
                    start, now_utc(), "FAILED", error_message=str(e)[:900])
            print(f"WARNING: OPTIMIZE {table} failed: {str(e)[:200]}")
else:
    print("OPTIMIZE skipped.")

[log] Silver-to-Gold / optimize:fact_commit -> SUCCESS (inserted=0, updated=0, quarantined=0, aggregated=0, seconds=6.9)
optimized: fact_commit
[log] Silver-to-Gold / optimize:fact_issue -> SUCCESS (inserted=0, updated=0, quarantined=0, aggregated=0, seconds=5.3)
optimized: fact_issue
[log] Silver-to-Gold / optimize:fact_pull_request -> SUCCESS (inserted=0, updated=0, quarantined=0, aggregated=0, seconds=6.1)
optimized: fact_pull_request
[log] Silver-to-Gold / optimize:fact_repo_daily_activity -> SUCCESS (inserted=0, updated=0, quarantined=0, aggregated=0, seconds=8.2)
optimized: fact_repo_daily_activity


## 7. Query plan of the biggest transformation (for the report)
`fact_repo_daily_activity` is the heaviest step: it reads three facts, groups them
five ways, and joins the results. Free Edition has no Spark UI, so the plan is
shown with `EXPLAIN FORMATTED`. Look for:
- `Exchange hashpartitioning(...)` = a **shuffle** (data moved between workers).
- `BroadcastHashJoin` = the small side was copied to every worker, no shuffle for it.
- `PushedFilters` / `DataFilters` in a scan = **predicate pushdown** (filter applied while reading files).

Also open this notebook's run in **Query History** (left bar, "Query History") and
take a screenshot of the Query Profile for the MERGE into `fact_repo_daily_activity`.

In [0]:
display(spark.sql("EXPLAIN FORMATTED " + sql_fact_repo_daily_activity()))

plan


In [0]:
# Predicate pushdown example: the incremental filter on a fact read.
display(spark.sql(f"""
EXPLAIN FORMATTED
SELECT sha, xxhash64(repo) AS repo_key, committed_at
FROM {T('silver_commit')}
WHERE load_timestamp >= TIMESTAMP '2026-10-01 00:00:00'"""))

plan
"== Physical Plan == PhotonResultStage (4) +- PhotonColumnarToRow (3) +- PhotonProject (2) +- PhotonScan parquet workspace.dav_project.silver_commit (1) (1) PhotonScan parquet workspace.dav_project.silver_commit Output [4]: [sha#36835, repo#36836, committed_at#36839, load_timestamp#36842] DictionaryFilters: [(load_timestamp#36842 >= 2026-10-01 00:00:00)] Location: PreparedDeltaFileIndex [s3://dbstorage-prod-iipnsih0xl/uc/1dc19bdf-4a14-43d2-b484-92f116be336a/b787fadc-03af-41e5-82d3-f45139cec5f2/__unitystorage/catalogs/88708ce5-8fc5-45dd-a6bb-c5c7b8520c41/tables/05709c13-2c49-499d-895a-34086c01186a] ReadSchema: struct RequiredDataFilters: [(load_timestamp#36842 >= 2026-10-01 00:00:00)] (2) PhotonProject Input [4]: [sha#36835, repo#36836, committed_at#36839, load_timestamp#36842] Arguments: [sha#36835, xxhash64(repo#36836, 42) AS repo_key#36818L, committed_at#36839] (3) PhotonColumnarToRow Input [3]: [sha#36835, repo_key#36818L, committed_at#36839] (4) PhotonResultStage Input [3]: [sha#36835, repo_key#36818L, committed_at#36839] == Photon Explanation == The query is fully supported by Photon. == Optimizer Statistics (table names per statistics state) == missing = partial = full = silver_commit"


## 8. Check: row counts and broken links

In [0]:
display(spark.sql(f"""
SELECT 'dim_repo' AS tbl, count(*) AS rows FROM {T('dim_repo')}
UNION ALL SELECT 'dim_date', count(*) FROM {T('dim_date')}
UNION ALL SELECT 'dim_contributor', count(*) FROM {T('dim_contributor')}
UNION ALL SELECT 'fact_commit', count(*) FROM {T('fact_commit')}
UNION ALL SELECT 'fact_issue', count(*) FROM {T('fact_issue')}
UNION ALL SELECT 'fact_pull_request', count(*) FROM {T('fact_pull_request')}
UNION ALL SELECT 'fact_repo_daily_activity', count(*) FROM {T('fact_repo_daily_activity')}
"""))

tbl,rows
dim_repo,9
dim_date,8401
dim_contributor,8128
fact_commit,13843
fact_issue,8838
fact_pull_request,18158
fact_repo_daily_activity,8599


In [0]:
# Every one of these should be 0. A number above 0 means a fact points at a dimension row that does not exist.
display(spark.sql(f"""
SELECT 'commit -> repo' AS link, count(*) AS broken FROM {T('fact_commit')} f LEFT JOIN {T('dim_repo')} d USING (repo_key) WHERE d.repo_key IS NULL
UNION ALL SELECT 'commit -> author', count(*) FROM {T('fact_commit')} f LEFT JOIN {T('dim_contributor')} d USING (author_key) WHERE d.author_key IS NULL
UNION ALL SELECT 'commit -> date', count(*) FROM {T('fact_commit')} f LEFT JOIN {T('dim_date')} d ON f.commit_date_key = d.date_key WHERE f.commit_date_key IS NOT NULL AND d.date_key IS NULL
UNION ALL SELECT 'issue -> repo', count(*) FROM {T('fact_issue')} f LEFT JOIN {T('dim_repo')} d USING (repo_key) WHERE d.repo_key IS NULL
UNION ALL SELECT 'issue -> author', count(*) FROM {T('fact_issue')} f LEFT JOIN {T('dim_contributor')} d USING (author_key) WHERE d.author_key IS NULL
UNION ALL SELECT 'pr -> repo', count(*) FROM {T('fact_pull_request')} f LEFT JOIN {T('dim_repo')} d USING (repo_key) WHERE d.repo_key IS NULL
UNION ALL SELECT 'pr -> author', count(*) FROM {T('fact_pull_request')} f LEFT JOIN {T('dim_contributor')} d USING (author_key) WHERE d.author_key IS NULL
"""))

link,broken
commit -> repo,0
commit -> author,0
commit -> date,0
issue -> repo,0
issue -> author,0
pr -> repo,0
pr -> author,0


In [0]:
display(spark.sql(f"""
    SELECT layer, entity, status, rows_inserted, rows_updated, rows_aggregated, duration_seconds, start_time
    FROM {T('pipeline_execution_logs')}
    WHERE run_id = '{RUN_ID}'
    ORDER BY start_time
"""))

layer,entity,status,rows_inserted,rows_updated,rows_aggregated,duration_seconds,start_time
Silver-to-Gold,dim_repo,SUCCESS,1,0,9,4.489818,2026-10-24T14:36:15.571Z
Silver-to-Gold,dim_date,SUCCESS,0,0,8401,3.875896,2026-10-24T14:36:21.791Z
Silver-to-Gold,dim_contributor,SUCCESS,3,0,8128,7.026359,2026-10-24T14:36:27.231Z
Silver-to-Gold,fact_commit,SUCCESS,3,13840,13843,5.516419,2026-10-24T14:36:35.698Z
Silver-to-Gold,fact_issue,SUCCESS,3,8835,8838,5.197361,2026-10-24T14:36:42.735Z
Silver-to-Gold,fact_pull_request,SUCCESS,2,18156,18158,6.757711,2026-10-24T14:36:49.363Z
Silver-to-Gold,fact_repo_daily_activity,SUCCESS,2,3,8599,8.815496,2026-10-24T14:36:57.697Z
Silver-to-Gold,optimize:fact_commit,SUCCESS,0,0,0,6.898154,2026-10-24T14:37:11.933Z
Silver-to-Gold,optimize:fact_issue,SUCCESS,0,0,0,5.321976,2026-10-24T14:37:20.362Z
Silver-to-Gold,optimize:fact_pull_request,SUCCESS,0,0,0,6.147384,2026-10-24T14:37:27.347Z


In [0]:
import json
dbutils.notebook.exit(json.dumps({
    "all_ok": not FAILED_STEPS,
    "problems": FAILED_STEPS,
    "run_id": RUN_ID,
    "tables": {k: v for k, v in results.items()},
}))

In [0]:
%sql
SELECT entity, status, rows_inserted, rows_updated, rows_quarantined, error_message
FROM workspace.dav_project.pipeline_execution_logs
WHERE layer = 'Bronze-to-Silver'
ORDER BY start_time DESC
LIMIT 6;

entity status rows_inserted rows_updated rows_quarantined error_message pulls SUCCESS 17930 226 0 null issues SUCCESS 8749 86 0 null commits FAILED 0 0 0 [DELTA_MERGE_UNRESOLVED_EXPRESSION] Cannot resolve has_extra_stats in UPDATE clause given columns s.sha, s.repo, s.author_key, s.committer_key, s.committed_at, s.message_length, s.is_merge_commit, s.load_timestamp.; line 2 pos 16

JVM stacktrace:
com.databricks.sql.transaction.tahoe.DeltaAnalysisException
	at com.databricks.sql.transaction.tahoe.ResolveDeltaMergeInto$.$anonfun$throwIfNotResolved$3(ResolveDeltaMergeInto.scala:57)
	at scala.collection.immutable.List.foreach(List.scala:323)
	at com.databricks.sql.transaction.tahoe.ResolveDeltaMergeInto$.throwIfNotResolved(ResolveDeltaMergeInto.scala:53)
	at com.databricks.sql.transaction.tahoe.ResolveDeltaMergeInto$.$anonfun$resolveOrFail$1(ResolveDeltaMergeInto.scala:74)
	at com.databricks.sql.transaction.tahoe.ResolveDeltaMergeInto$.$anonfun$resolveOrFail$1$adapted(ResolveDeltaMergeInto.scala:74)
	at scala.collection.immutable.List.foreach(List.scala:323)
	at com.databricks.sql.transaction.tahoe.ResolveDeltaMergeInto$.resolveOrFail(ResolveDeltaMergeInto.scala:74)
	at com.databricks.sql.transaction.tahoe.ResolveDeltaMergeInto$.resolveStar(ResolveDeltaMergeInto.scala:739)
	at com.databricks.sql.transaction.tahoe.ResolveDeltaMergeInto$.$anonfun$resolveClause$4(ResolveDeltaMergeInto.scala:316)
	at scala.collection.immutable.List.flatMap(List.scala:283)
	at scala.collection.immutable.List.flatMap(List.scala:79)
	at com.databricks.sql.transaction.tahoe.ResolveDeltaMergeInto$.resolveClause(ResolveDeltaMergeInto.scala:311)
	at com.databricks.sql.transaction.tahoe.ResolveDeltaMergeInto$.$anonfun$resolveReferencesAndSchema$1(ResolveDeltaMergeInto.scala:941)
	at scala.collection.immutable.List.map(List.scala:236)
	at scala.collection.immutable.List.map(List.scala:79)
	at com.databricks.sql.transaction.tahoe.ResolveDeltaMergeInto$.resolveReferencesAndSchema(ResolveDeltaMergeInto.scala:939)
	at com.databricks.sql.transaction.tahoe.DeltaAnalysis$$anonfun$apply$1.applyOrElse(DeltaAnalysis.scala:1665)
	at com.databricks.sql.transaction.tahoe.DeltaAnalysis$$anonfun$apply$1.applyOrElse(DeltaAnalysis.scala:166)
	at org.apache.spark.sql.catalyst.plans.logical.AnalysisHelper.$anonfun$resolveOperatorsDownWithPruning$2(AnalysisHelper.scala:208)
	at org.apache.spark.sql.catalyst.trees.CurrentOrigin$.withOrigin(origin.scala:142)
	at org.apache.spark.sql.catalyst.plans.logical.AnalysisHelper.$anonfun$resolveOperatorsDownWithPruning$1(AnalysisHelper.scala:208)
	at org.apache.spark.sql.catalyst.plans.logical.AnalysisHelper$.allowInvokingTransformsInAnalyzer(AnalysisHelper.scala:441)
	at org.apache.spark.sql.catalyst.plans.logical.AnalysisHelper.resolveOperatorsDownWithPruning(AnalysisHelper.scala:206)
	at org.apache.spark.sql.catalyst.plans.logical.AnalysisHelper.resolveOperatorsDownWithPruning$(AnalysisHelper.scala:202)
	at org.apache.spark.sql.catalyst.plans.logical.LogicalPlan.resolveOperatorsDownWithPruning(LogicalPlan.scala:49)
	at org.apache.spark.sql.catalyst.plans.logical.AnalysisHelper.resolveOperatorsDown(AnalysisHelper.scala:198)
	at org.apache.spark.sql.catalyst.plans.logical.AnalysisHelper.resolveOperatorsDown$(AnalysisHelper.scala:197)
	at org.apache.spark.sql.catalyst.plans.logical.LogicalPlan.resolveOperatorsDown(LogicalPlan.scala:49)
	at com.databricks.sql.transaction.tahoe.DeltaAnalysis.apply(DeltaAnalysis.scala:166)
	at com.databricks.sql.transaction.tahoe.DeltaAnalysis.apply(DeltaAnalysis.scala:160)
	at org.apache.spark.sql.catalyst.rules.RuleExecutor.$anonfun$execute$18(RuleExecutor.scala:572)
	at org.apache.spark.sql.catalyst.rules.RecoverableRuleExecutionHelper.processRule(RuleExecutor.scala:758)
	at org.apache.spark.sql.catalyst.rules.RecoverableRuleExecutionHelper.processRule$(RuleExecutor.scala:725)
	at org.apache.spark.sql.catalyst.rules.RuleExecutor.processRule(RuleExecutor.scala:156)
	at org.apache.spark.sql.catalyst.rules.Ru

In [0]:
%sql
SELECT repo, count(*) AS commits
FROM workspace.dav_project.silver_commit
GROUP BY repo;

repo,commits
matplotlib/matplotlib,1
pytest-dev/pytest,2
scipy/scipy,1
pandas-dev/pandas,36
demo/fake-repo,3


In [0]:
%sql
SELECT * FROM workspace.dav_project.dim_repo;

repo_key,repo_full_name,owner,repo_name
-1120679421055069871,matplotlib/matplotlib,matplotlib,matplotlib
-4973909499231263495,pytest-dev/pytest,pytest-dev,pytest
-3181326774358190712,scipy/scipy,scipy,scipy
1076036374452082389,pandas-dev/pandas,pandas-dev,pandas
-3810921423041699768,demo/fake-repo,demo,fake-repo
-887624950100305163,pallets/flask,pallets,flask
-8082243418927947421,fastapi/fastapi,fastapi,fastapi
674668530088994642,numpy/numpy,numpy,numpy
-7370082827790465918,scikit-learn/scikit-learn,scikit-learn,scikit-learn
